# 01 — Exploration du dataset médical bilingue

Ce notebook explore les jeux de données préparés en **Semaine 1** (SFT, DPO,
évaluation clinique). Il sert à *vérifier la qualité* et la *composition* du
corpus avant l'entraînement.

> Pré-requis : avoir exécuté le pipeline de données (`scripts/run_pipeline.sh`
> ou les commandes `triage-collect` → `triage-split`).

In [ ]:
# On rend le paquet `triage` importable depuis le notebook.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

from triage.config import SFT_DIR, DPO_DIR, EVAL_DIR
from triage.data.stats import sft_stats
from triage.utils.common import read_jsonl

## 1. Composition du jeu d'entraînement SFT

In [ ]:
stats = sft_stats(SFT_DIR / 'train.jsonl')
print('Total exemples :', stats['total'])
print('Par langue     :', stats['par_langue'])
print('Par tâche      :', stats['par_tache'])
print('Par source     :', stats['par_source'])
print('Par niveau de triage :', stats['par_niveau_triage'])

In [ ]:
# Visualisation rapide de la répartition par langue (corpus bilingue)
import matplotlib.pyplot as plt

lang = stats['par_langue']
plt.bar(lang.keys(), lang.values(), color=['#4C72B0', '#DD8452'])
plt.title('Répartition des exemples SFT par langue')
plt.ylabel('Nombre d\'exemples')
plt.show()

## 2. Aperçu d'un exemple de triage (format cible)

In [ ]:
for r in read_jsonl(SFT_DIR / 'train.jsonl'):
    if r['task_type'] == 'triage':
        print('INSTRUCTION :', r['instruction'][:200])
        print('\nRÉPONSE :\n', r['output'][:600])
        print('\nMÉTADONNÉES :', r['metadata'])
        break

## 3. Vérification de l'anonymisation (RGPD)

In [ ]:
# On compte les enregistrements marqués comme anonymisés.
anon = sum(1 for r in read_jsonl(SFT_DIR / 'train.jsonl')
           if r.get('metadata', {}).get('anonymized'))
print(f'Enregistrements anonymisés : {anon} / {stats["total"]}')

## 4. Aperçu d'une paire préférentielle (DPO)

In [ ]:
for r in read_jsonl(DPO_DIR / 'train.jsonl'):
    if r['task_type'] == 'triage':
        print('PROMPT   :', r['prompt'][:150])
        print('\nCHOSEN   :', r['chosen'][:250])
        print('\nREJECTED :', r['rejected'][:250])
        break